# 07 — Inference Comparison: Mamba vs GPT-2

This notebook compares inference behavior between Mamba (`mamba-130m-hf`) and GPT-2 (`gpt2`, 124M params) using measured GPU benchmark data.

**What you will learn:**
1. How Mamba's recurrent state compares to GPT-2's KV cache in practice
2. TTFT (time to first token) and decode throughput differences
3. Memory scaling behavior as context length grows
4. What these results mean (and what they don't)


## Why Compare Mamba and GPT-2?

Both models are ~130M parameters, making them a fair size comparison. The key architectural difference:

| | GPT-2 (Transformer) | Mamba (SSM) |
|---|---|---|
| Attention mechanism | Quadratic self-attention | Selective state space recurrence |
| Context state | KV cache (grows with length) | Fixed-size recurrent state |
| Prefill | Parallel (single matmul) | Sequential (scan over tokens) |
| Decode | Fast (cached) | Fast (fixed state update) |

**The central claim:** Mamba's memory usage should grow more slowly with context length because it maintains a fixed-size state instead of a growing KV cache.

**Important caveat:** The HuggingFace Mamba implementation uses a pure Python sequential loop (no optimized CUDA kernels). This makes Mamba's *throughput* slower than GPT-2 in these benchmarks. The optimized `mamba-ssm` package would close this gap, but we are benchmarking our from-scratch implementation path.


## Load Measured GPU Results

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Load inference benchmark results
inf_path = Path("../benchmarks/results/inference_results.gpu.json")
if not inf_path.exists():
    inf_path = Path("benchmarks/results/inference_results.gpu.json")

# Load memory scaling results (dedicated sweep)
mem_path = Path("../benchmarks/results/memory_scaling.gpu.json")
if not mem_path.exists():
    mem_path = Path("benchmarks/results/memory_scaling.gpu.json")

inf_data = json.loads(inf_path.read_text())
mem_data = json.loads(mem_path.read_text())

print(f"Inference results: {len(inf_data)} rows")
print(f"Memory scaling: {len(mem_data)} rows")
print(f"Hardware: {inf_data[0]['machine']}")

## TTFT and Decode Throughput

In [ ]:
# Group inference results by model
models = {}
for row in inf_data:
    name = row["model_name"]
    if name not in models:
        models[name] = {"prompts": [], "ttft": [], "throughput": [], "memory": []}
    models[name]["prompts"].append(row["prompt_tokens"])
    models[name]["ttft"].append(row["ttft_ms"])
    models[name]["throughput"].append(row["throughput_tokens_per_s"])
    models[name]["memory"].append(row["peak_memory_mb"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {"state-spaces/mamba-130m-hf": "#4C78A8", "gpt2": "#F58518"}
labels = {"state-spaces/mamba-130m-hf": "Mamba-130M", "gpt2": "GPT-2 (124M)"}

for name, stats in models.items():
    c = colors.get(name, "gray")
    lbl = labels.get(name, name)
    axes[0].plot(stats["prompts"], stats["ttft"], "o-", label=lbl, color=c, linewidth=2)
    axes[1].plot(stats["prompts"], stats["throughput"], "o-", label=lbl, color=c, linewidth=2)
    axes[2].plot(stats["prompts"], stats["memory"], "o-", label=lbl, color=c, linewidth=2)

axes[0].set_title("Time to First Token")
axes[0].set_xlabel("Prompt tokens")
axes[0].set_ylabel("TTFT (ms)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Decode Throughput")
axes[1].set_xlabel("Prompt tokens")
axes[1].set_ylabel("Tokens/s")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Peak Memory")
axes[2].set_xlabel("Prompt tokens")
axes[2].set_ylabel("Memory (MB)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle("Mamba vs GPT-2 Inference (NVIDIA A10G)", fontsize=13)
fig.tight_layout()
plt.show()

## Memory Scaling: The Key Chart

This is the chart that tells the systems story. We ran a dedicated memory sweep with precise token counts to compare how peak memory grows with context length:


In [ ]:
# Parse memory scaling data
mem_models = {}
for row in mem_data:
    name = row.get("model", row.get("model_name", "unknown"))
    if name not in mem_models:
        mem_models[name] = {"tokens": [], "memory": []}
    mem_models[name]["tokens"].append(row.get("actual_prompt_tokens", row.get("prompt_tokens", 0)))
    mem_models[name]["memory"].append(row["peak_memory_mb"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Absolute memory
for name, stats in mem_models.items():
    pairs = sorted(zip(stats["tokens"], stats["memory"]))
    xs, ys = zip(*pairs)
    c = colors.get(name, "gray")
    lbl = labels.get(name, name)
    axes[0].plot(xs, ys, "o-", label=lbl, color=c, linewidth=2)

axes[0].set_title("Peak Memory vs Context Length")
axes[0].set_xlabel("Prompt tokens")
axes[0].set_ylabel("Peak memory (MB)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Memory growth rate (delta from baseline)
for name, stats in mem_models.items():
    pairs = sorted(zip(stats["tokens"], stats["memory"]))
    xs, ys = zip(*pairs)
    baseline = ys[0]
    deltas = [y - baseline for y in ys]
    c = colors.get(name, "gray")
    lbl = labels.get(name, name)
    axes[1].plot(xs, deltas, "o-", label=lbl, color=c, linewidth=2)

axes[1].set_title("Memory Growth Above Baseline")
axes[1].set_xlabel("Prompt tokens")
axes[1].set_ylabel("Additional memory (MB)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Memory Scaling Comparison (NVIDIA A10G)", fontsize=13)
fig.tight_layout()
plt.show()

## Results Summary Table

In [ ]:
print("=" * 85)
print(f"{'Model':<30} {'Prompt':>8} {'TTFT(ms)':>10} {'tok/s':>8} {'Memory(MB)':>12}")
print("=" * 85)
for row in sorted(inf_data, key=lambda r: (r["model_name"], r["prompt_tokens"])):
    print(f"{row['model_name']:<30} {row['prompt_tokens']:>8} "
          f"{row['ttft_ms']:>10.1f} {row['throughput_tokens_per_s']:>8.1f} "
          f"{row['peak_memory_mb']:>12.1f}")
print("=" * 85)

## Interpreting the Results

### What the data shows

1. **GPT-2 has faster TTFT** because its prefill is a single parallel attention pass, while HF Mamba processes tokens sequentially through a Python loop.

2. **GPT-2 has higher decode throughput** for the same reason — the HF Mamba implementation lacks optimized CUDA kernels. With `mamba-ssm` installed, Mamba's throughput would be competitive or better.

3. **Memory scaling differs**: Both models show memory growth with context length, but the growth patterns differ due to KV cache (GPT-2) vs sequential scan intermediates (Mamba HF).

### What this means for the systems argument

The core Mamba advantage — **O(1) state vs O(L) KV cache** — becomes decisive at long contexts (thousands to millions of tokens). At the short contexts we can test here (limited by GPT-2's 1024 position limit), the difference is visible but not dramatic.

The real systems takeaway from this repo is at the **kernel level**: the fused Triton scan kernel is 70-120x faster than the PyTorch reference, and its performance is memory-bound (as shown in the profiling notebook). This is the bottleneck that matters for Mamba inference.

### Honest limitations

- HF Mamba without optimized CUDA kernels is not representative of production Mamba performance
- GPT-2's 1024 token limit prevents testing at the long contexts where Mamba's advantage is most visible
- Both models are small (~130M) — larger models would show more pronounced differences
